# Notebook 01 — Exploratory Data Analysis (EDA)

**Goal:** Understand the dataset before building any model.

The most important thing you will see: **class imbalance**.
Some endangerment categories have hundreds of languages; others have a handful.
This imbalance is the core problem we spend the rest of the project solving.

Run cells top to bottom.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath('..'))

from src.load_data import load_raw, LABEL_COL
from src.preprocess import clean, ENDANGERMENT_ORDER, save_processed
from src.visualise import plot_class_distribution

print('Imports OK')

In [1]:
# ── Load raw data ─────────────────────────────────────────────────────
# If this raises FileNotFoundError, see README.md for the download link.
df_raw = load_raw()
print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')

NameError: name 'load_raw' is not defined

In [ ]:
# ── Preview the data ──────────────────────────────────────────────────
# Look at the columns: language name, country, family, speakers, label
df_raw.head(10)

In [ ]:
# ── THE KEY PLOT: class distribution ─────────────────────────────────
# This makes the imbalance immediately visible.
# 'Safe' has far more languages than 'Extinct'.
# A naive model trained on this will just predict 'Safe' for everything.

print('Label value counts:')
print(df_raw[LABEL_COL].value_counts())

plot_class_distribution(
    df_raw[LABEL_COL],
    title='Languages per endangerment level (raw data)',
    save_name='01_class_distribution_raw'
)

In [ ]:
# ── Missing values ────────────────────────────────────────────────────
# How much data is missing per column?
missing = df_raw.isnull().sum()
pct = (missing / len(df_raw) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_%': pct}).query('missing_count > 0')

In [ ]:
# ── Clean the data ────────────────────────────────────────────────────
# clean() is defined in src/preprocess.py — open it and read the comments.
# It fills missing values, standardises labels, and encodes categories.
df_clean = clean(df_raw)
print(f'After cleaning: {df_clean.shape}')

In [ ]:
# ── Speaker count distribution (log scale) ────────────────────────────
# log10 scale because the range is enormous: 0 speakers to 1 billion+
speakers = df_clean['Number of speakers'].replace(0, 0.1)
plt.figure(figsize=(9, 4))
plt.hist(np.log10(speakers), bins=40, color='#5B8DB8', edgecolor='white')
plt.xlabel('log10(speakers)  [0=1, 3=1,000, 6=1,000,000]')
plt.ylabel('Number of languages')
plt.title('Speaker count distribution (log scale)')
plt.tight_layout()
plt.savefig('../outputs/figures/01_speaker_dist.png', dpi=150)
plt.show()

In [ ]:
# ── Save cleaned data ─────────────────────────────────────────────────
# Other notebooks load from data/processed/ instead of re-cleaning.
save_processed(df_clean)
print('Notebook 01 complete. Proceed to 02_baseline.ipynb')